# GenBloom

Loads **GenBloom-V** and **GenBloom-G** from HuggingFace and runs inference on one example patient.

- Model weights: [`MarrLab/GenBloom`](https://huggingface.co/MarrLab/GenBloom)
- Patient embeddings: [`MarrLab/DinoBloom_hemato_embeddings`](https://huggingface.co/datasets/MarrLab/DinoBloom_hemato_embeddings)

Inputs to both models are per-patient DinoBloom-B cell embeddings of shape `(N_cells, 768)`. Outputs are a single 768-d patient embedding.

## 1. Imports

In [12]:
import sys
from pathlib import Path

import h5py
import torch
from huggingface_hub import hf_hub_download
# sys.path.append("") # append your working dirextory if needed

from genbloom_g import _build_vision_model, load_contrastive_model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

device: cuda


## 2. Download checkpoints and one example patient

Cached under `~/.cache/huggingface/`.

In [8]:
MODEL_REPO = "MarrLab/GenBloom"
DATA_REPO  = "MarrLab/DinoBloom_hemato_embeddings"

v_ckpt_path = hf_hub_download(MODEL_REPO, "checkpoints/genbloom_v/genbloom_v.pth")
g_ckpt_path = hf_hub_download(MODEL_REPO, "checkpoints/genbloom_g/genbloom_g_fold0.pth")

patient_h5 = hf_hub_download(
    DATA_REPO,
    "patient_embeddings/aml_hehr/DHA.h5",
    repo_type="dataset",
)

print("genbloom_v:", v_ckpt_path)
print("genbloom_g:", g_ckpt_path)
print("patient   :", patient_h5)

genbloom_v: /ictstr01/groups/labs/marr/qscd01/cache_labwide/huggingface/hub/models--MarrLab--GenBloom/snapshots/791c7046e5a7e167afa18ad3483c4b2f8f10cd6e/checkpoints/genbloom_v/genbloom_v.pth
genbloom_g: /ictstr01/groups/labs/marr/qscd01/cache_labwide/huggingface/hub/models--MarrLab--GenBloom/snapshots/791c7046e5a7e167afa18ad3483c4b2f8f10cd6e/checkpoints/genbloom_g/genbloom_g_fold0.pth
patient   : /ictstr01/groups/labs/marr/qscd01/cache_labwide/huggingface/hub/datasets--MarrLab--DinoBloom_hemato_embeddings/snapshots/ffb086ef7766f372cbba1ffc43089813812f893c/patient_embeddings/aml_hehr/DHA.h5


## 3. Load the patient cell-embedding bag

In [9]:
with h5py.File(patient_h5, "r") as f:
    features = torch.from_numpy(f["features"][:])  # (N_cells, 768)
    label    = int(f["labels"][()])

print("features:", tuple(features.shape), features.dtype)
print("label   :", label)

features: (496, 768) torch.float32
label   : 3


## 4. GenBloom-V inference

Vision encoder only — aggregates the cell-embedding bag into a single 768-d CLS token.

In [10]:
v_model = _build_vision_model(
    checkpoint_path=v_ckpt_path,
    embed_dim=768, feature_dim=768, depth=6, num_heads=12,
    patch_size=1, device=DEVICE,
)

with torch.no_grad():
    x = features.unsqueeze(0).to(DEVICE)
    v_embedding = v_model.forward_features(x)["x_norm_clstoken"].squeeze(0).cpu()

print("GenBloom-V patient embedding:", tuple(v_embedding.shape))
print(v_embedding[:8])

GenBloom-V patient embedding: (768,)
tensor([ 0.4923,  0.7706, -0.1021, -0.9498,  0.2268, -0.3003,  0.1340,  0.5965])


## 5. GenBloom-G inference

Genetically-aligned model: GenBloom-V backbone (fine-tuned) + contrastive projection head. We use the unprojected 768-d CLS token, which matches the evaluation setup in the paper.

In [11]:
g_model, projection_dim = load_contrastive_model(
    checkpoint_path=g_ckpt_path,
    dinov2_checkpoint_path=v_ckpt_path,
    embed_dim=768, feature_dim=768, depth=6, num_heads=12,
    device=DEVICE,
)
g_model.return_unprojected = True  # use 768-d CLS, bypass projection head

with torch.no_grad():
    x = features.unsqueeze(0).to(DEVICE)
    g_embedding = g_model.forward_features(x)["x_norm_clstoken"].squeeze(0).cpu()

print("GenBloom-G patient embedding:", tuple(g_embedding.shape))
print("projection_dim (unused here):", projection_dim)
print(g_embedding[:8])

GenBloom-G patient embedding: (768,)
projection_dim (unused here): 128
tensor([ 2.1282,  0.1467,  0.4177,  0.2970,  0.5371, -0.2768,  0.5932, -0.2537])
